In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

np.set_printoptions(suppress=True, precision=4)

In [2]:
train = pd.read_csv('train.csv')
train = train.round(4) # precision = 4
test = pd.read_csv('test.csv')
train.head()
#train.describe()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [4]:
train[train['Sex'] == 'male'].Age.describe()
#print(train[train['Sex'] == 'male'].Age.mode())
# mode is 19 / 25, mean is 30

count    453.000000
mean      30.726645
std       14.678201
min        0.420000
25%       21.000000
50%       29.000000
75%       39.000000
max       80.000000
Name: Age, dtype: float64

In [5]:
train[train['Sex'] == 'female'].Age.describe()
#train[train['Sex'] == 'female'].Age.mode()
# mode is 24, mean is 28

count    261.000000
mean      27.915709
std       14.110146
min        0.750000
25%       18.000000
50%       27.000000
75%       37.000000
max       63.000000
Name: Age, dtype: float64

In [6]:
train['Embarked']

0      S
1      C
2      S
3      S
4      S
      ..
886    S
887    S
888    S
889    C
890    Q
Name: Embarked, Length: 891, dtype: object

In [7]:
#Age和Cabin还有Embarked有缺失值
#Age使用众数或者均值填充（分男女）, carbin舍弃，embark随便填
train['Age'] = train.groupby('Sex')['Age'].transform(
    lambda x: x.fillna(x.median())  # 或 x.mean()
)
train['Embarked'] = train['Embarked'].fillna('S')

#transform gender to num
train['Sex_num'] = train['Sex'].map({'male': 0, 'female': 1})

#transform embarked to num
train['Embarked'] = train['Embarked'].map({'C': 0, 'S': 1, 'Q': 2})

#minmaxscaler
mm_scaler = MinMaxScaler()
train['Age'] = mm_scaler.fit_transform(train[['Age']])
train['Fare'] = mm_scaler.fit_transform(train[['Fare']])

In [8]:
train['Embarked'].isnull().sum()

np.int64(0)

In [9]:
features = ['Pclass', 'Sex_num', 'Age', 'SibSp', 'Fare', 'Embarked']
#Too many NaN in Carbin, can't use it
# SibSp: 兄弟姐妹+配偶, Parch: parent + children, Embarked: 登船港口
target = ['Survived']

train_data = train[features]

X = train[['PassengerId'] + features].values
print(X)

#将各特征提取，转换成向量


[[  1.       3.       0.     ...   1.       0.0142   1.    ]
 [  2.       1.       1.     ...   1.       0.1391   0.    ]
 [  3.       3.       1.     ...   0.       0.0155   1.    ]
 ...
 [889.       3.       1.     ...   1.       0.0458   1.    ]
 [890.       1.       0.     ...   0.       0.0586   0.    ]
 [891.       3.       0.     ...   0.       0.0151   2.    ]]


In [10]:
#-----------------model_1: linear classifier, Centroid method-----------#
